In [1]:
#先引入LLM
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

C:\Users\13407\.conda\envs\python310\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 1、定义工具

##### 这个相较于v0.3没有变化。

In [3]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

tools = [
    search,
    get_weather ]
agent = create_agent(model, tools=tools)

## 2、工具错误处理

##### 要自定义工具错误的处理方式，请使用 @wrap_tool_call 装饰器创建中间件

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

agent = create_agent(
    model=model,
    tools=[search, get_weather],
    middleware=[handle_tool_errors]
)

##### 其实就是一个工具错误提示，当传入工具（也作为参数）和参数（也是传入工具的参数）时，验证是否正常

## 3、系统提示

##### 可以通过提供提示来塑造代理处理任务的方式。system_prompt 参数可以作为字符串提供

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

## 4、动态系统提示

##### 对于需要根据运行时上下文或代理状态修改系统提示的更高级用例，可以使用中间件。@dynamic_prompt装饰器创建中间件，根据模型请求动态生成系统提示

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""

    user_role = request.runtime.context.get("user_role", "user")
    #从字典中获取"user_role"键对应的值，如果这个键不存在，就返回默认值"user"。
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model=model,
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

##### 这段代码就是基于result中的user_role传入中间件的函数user_role_prompt中，改变对话风格。
##### 在 LangChain 中，中间件访问的是 request.runtime.context，而这个 context 是通过 agent.invoke() 的 context 参数显式传入的。
##### request.runtime.context 是 LangChain 中一个在请求处理过程中传递数据的容器

## 5、结构化输出

##### 在某些情况下，您可能希望代理以特定格式返回输出。LangChain 通过 response_format 参数提供结构化输出策略。

### （1）工具策略

##### ToolStrategy使用人工工具调用来生成结构化输出。这适用于任何支持工具调用的模型

In [3]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

### （2）提供商策略

##### ProviderStrategy使用模型提供程序的本机结构化输出生成。这更可靠，但仅适用于支持原生结构化输出的提供商（例如 OpenAI）

In [6]:
from langchain.agents.structured_output import ProviderStrategy

agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


##### “原生结构化输出”指的是这种能力是由大模型本身在底层直接提供的，而不是通过后处理或巧妙的 Prompt 工程实现的。
##### “支持原生结构化输出的提供商”就是指该提供商提供的 API 接口 中，包含了让你直接指定输出 JSON Schema 的参数。目前，并不是所有提供商都支持这个功能。

## 6、记忆

##### 代理通过消息状态自动维护对话历史记录。您还可以将代理配置为使用自定义状态架构来记住对话期间的其他信息。存储在状态中的信息可以被认为是智能体的短期记忆：
##### 自定义状态架构必须将 AgentState 扩展为 .TypedDict
##### 有两种方法可以定义自定义状态：
##### 1.通过中间件（首选）
##### 2.通过 state_schema on create_agent

### (1)通过中间件定义状态

##### 当需要通过附加到所述中间件的特定中间件钩子和工具访问自定义状态时，使用中间件定义自定义状态。从中我们看出使用中间件可以提供更多的自定义状态

In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware


class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = []  #[tool1, tool2]

    def before_model(self, state: CustomState, runtime) -> dict[str] | None:
        ...

agent = create_agent(
    model,
    tools=tools,
    middleware=[CustomMiddleware()]
)

# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

### （2）通过定义状态state_schema

##### 使用 state_schema 参数作为快捷方式来定义仅在工具中使用的自定义状态。没有中间件只会将字符串传给大模型，无法进行更多自定义

In [ ]:
from langchain.agents import AgentState


class CustomState(AgentState):
    user_preferences: dict

agent = create_agent(
    model,
    tools=[],                           #[tool1, tool2],
    state_schema=CustomState
)
# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

## 7、流

##### 我们已经了解了如何调用代理以获得最终响应。如果代理执行多个步骤，这可能需要一段时间。为了显示中间进度，我们可以在消息发生时流式传输它们。这与v0.3对流的定义一样

In [ ]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

## 8、中间件

##### 中间件为在执行的不同阶段自定义代理行为提供了强大的可扩展性。您可以使用中间件来：
##### （1）调用模型之前的进程状态（例如，消息修剪、上下文注入）
##### （2）修改或验证模型的响应（例如，护栏、内容过滤）
##### （3）使用自定义逻辑处理工具执行错误
##### （4）根据状态或上下文实现动态模型选择
##### （5）添加自定义日志记录、监视或分析
##### 中间件无缝集成到代理的执行图中，允许您在关键点截获和修改数据流，而无需更改核心代理逻辑。
##### 中间件算是langchain在1.0版本引入的新知识。在不更改代理逻辑的基础上，可以对中间的过程高度自定义。详细的在后续进行学习。